# Document search with Gemini embeddings

In [ ]:
%pip install -U -q "google-genai>=1.0.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.1/246.1 kB 27.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.53.0 which is incompatible.
google-cloud-aiplatform 1.148.1 requires google-genai<2.0.0,>=1.66.0; python_version >= "3.10", but you have google-genai 2.4.0 which is incompatible.
google-adk 1.29.0 requires google-genai<2.0.0,>=1.64.0, but you have google-genai 2.4.0 which is incompatible.


To run the following cell, your API key must be stored it in a Colab Secret named `GOOGLE_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see the [Authentication](https://github.com/google-gemini/cookbook/blob/main/quickstarts/Authentication.ipynb) quickstart for an example.

In [ ]:
from google import  genai
from google.genai import types
from google.colab import userdata

GEMINI_API_KEY=userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=GEMINI_API_KEY)

### Read data

In [2]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Data.csv')

### Get the embeddings for question text

In [ ]:
# Get the embeddings of each text and add to an embeddings column in the dataframe

EMBEDDING_MODEL_ID = MODEL_ID = "gemini-embedding-001"  # @param ["gemini-embedding-001", "gemini-embedding-2-preview"] {"allow-input": true, "isTemplate": true}

def embed_fn(column):
  response = client.models.embed_content(
        model=EMBEDDING_MODEL_ID,
        contents=column,
        config=types.EmbedContentConfig(
            task_type="retrieval_query"
        )
    )

  return response.embeddings[0].values

df['Qn_embeddings'] = df.apply(lambda row: embed_fn(row['Question']), axis=1)

### Ingest data into Neo4j Graph DB

In [ ]:
!pip install neo4j

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 18.9 MB/s eta 0:00:00


In [ ]:
from neo4j import GraphDatabase

NEO4J_URI = "neo4j+s://9c2d34fb.databases.neo4j.io"
NEO4J_AUTH = ("neo4j", "ibbOTMLSHuIzGwzXCtuOjEgZMSNv4TOrrTM5lR2PTeg")

def save_to_neo4j(question_id, vector):

    query = """
    MERGE (q:Qn {`Question ID`: $question_id})
    SET q.embedding = $embedding
    RETURN q.`Question ID` AS question_id,
           size(q.embedding) AS dimensions
    """

    driver = GraphDatabase.driver(
        NEO4J_URI,
        auth=NEO4J_AUTH
    )

    with driver.session(database="neo4j") as session:

        result = session.run(
            query,
            question_id=question_id,
            embedding=[float(x) for x in vector]
        )

        record = result.single()

    driver.close()

# Apply function
save_to_neo4j(question_id, vector)

Saved Question ID: 108


In [ ]:
# Create a vector index
CREATE VECTOR INDEX qnembeddingIndex IF NOT EXISTS
FOR (q:Qn)
ON q.embedding
OPTIONS { indexConfig: {
 `vector.dimensions`: 3072,
 `vector.similarity_function`: 'cosine'
}}